# 01 Tools

This notebook exercises the examples from `01_Fundamentals/README.md`, starting at **Building and Orchestrating Tools**.

Notes:
- The tool-building examples are kept close to the README.
- The README agent examples use older LangChain helpers such as `initialize_agent(...)` and named agent types.
- In this environment, LangChain v1 exposes `create_agent(...)` instead, so the agent section below uses the current API while covering the same ideas with **OpenAI models only**.
- Add `OPENAI_API_KEY` to your `.env` file before running the model-backed cells.


In [1]:
import os
import re
from typing import Dict, List, Union

from dotenv import load_dotenv
from langchain_core.tools import Tool, StructuredTool, tool

load_dotenv()

True

## 1. Building Tools

We start with the simple string-based tools from the README, then move to structured tools with typed inputs.

In [2]:
def add_numbers(inputs: str) -> dict:
    """
    Adds all integer numbers found in a string.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces or text.

    Returns:
    - dict: A dictionary with key 'result' containing the sum of extracted integers.
    """
    digits = [int(x) for x in inputs.split() if x.isdigit()]
    return {"result": sum(digits)}


add_tool = Tool.from_function(
    func=add_numbers,
    name="add_numbers",
    description="Adds numbers from a string input.",
)

print(add_tool.invoke("10 20 30"))
print(add_tool.name)
print(add_tool.description)
print(getattr(add_tool, "args", None))


{'result': 60}
add_numbers
Adds numbers from a string input.
{'tool_input': {'type': 'string'}}


In [3]:
@tool
def add_numbers_regex(inputs: str) -> dict:
    """
    Extracts and sums all integers from a string using regex.

    Parameters:
    - inputs (str): A string possibly containing numbers.

    Returns:
    - dict: A dictionary with key 'result' containing the sum.
    """
    numbers = [int(x) for x in re.findall(r"\d+", inputs)]
    return {"result": sum(numbers)}


print(add_numbers_regex.invoke("The numbers are 10 and 20"))
print(add_numbers_regex.name)
print(add_numbers_regex.description)
print(getattr(add_numbers_regex, "args", None))


{'result': 30}
add_numbers_regex
Extracts and sums all integers from a string using regex.

Parameters:
- inputs (str): A string possibly containing numbers.

Returns:
- dict: A dictionary with key 'result' containing the sum.
{'inputs': {'title': 'Inputs', 'type': 'string'}}


In [4]:
@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Sums a list of numbers, optionally using absolute values.

    Parameters:
    - numbers (List[float]): List of numbers to sum.
    - absolute (bool): If True, sums absolute values of the numbers.

    Returns:
    - float: The computed sum.
    """
    if absolute:
        numbers = [abs(x) for x in numbers]
    return sum(numbers)


result = add_numbers_with_options.invoke({
    "numbers": [-1.2, -5.0],
    "absolute": True,
})

print(result)
print(add_numbers_with_options.name)
print(add_numbers_with_options.description)
print(add_numbers_with_options.args)
print(getattr(add_numbers_with_options, "args_schema", None))


6.2
add_numbers_with_options
Sums a list of numbers, optionally using absolute values.

Parameters:
- numbers (List[float]): List of numbers to sum.
- absolute (bool): If True, sums absolute values of the numbers.

Returns:
- float: The computed sum.
{'numbers': {'items': {'type': 'number'}, 'title': 'Numbers', 'type': 'array'}, 'absolute': {'default': False, 'title': 'Absolute', 'type': 'boolean'}}
<class 'langchain_core.utils.pydantic.add_numbers_with_options'>


In [5]:
def safe_add(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Attempts to sum integers in a string; returns an error message if none are found.

    Parameters:
    - inputs (str): Input string containing numbers.

    Returns:
    - Dict[str, Union[float, str]]:
        - 'result' (float): Sum if numbers are found.
        - 'result' (str): Error message if no numbers are found.
    """
    numbers = [int(x) for x in inputs.split() if x.isdigit()]
    if not numbers:
        return {"result": "No numbers found"}
    return {"result": sum(numbers)}


print(safe_add("10 20 30"))
print(safe_add("no numbers here"))


{'result': 60}
{'result': 'No numbers found'}


## 2. Building and Orchestrating Agents with OpenAI

The README shows three agent styles:
- string-based tools
- structured tools with typed inputs
- OpenAI-native structured behavior

In this LangChain version, the unified `create_agent(...)` API replaces the older `initialize_agent(...)` patterns. The cells below cover the same progression with `ChatOpenAI`.

In [6]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4.1-nano"

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set. Add it to .env before running the LLM cells.")

# A low-cost OpenAI model is enough for these tool-calling demos.
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def print_last_message(state: dict) -> None:
    """Pretty-print the final message returned by a LangChain agent graph."""
    last_message = state["messages"][-1]
    print(last_message.content if hasattr(last_message, "content") else last_message)


response = llm.invoke("What is 2 + 2?")
print(response.content)


2 + 2 equals 4.


In [7]:
# This mirrors the README's fragile string-tool example.
# We intentionally use integers here because the toy parser ignores decimals.
string_tool_agent = create_agent(
    model=llm,
    tools=[add_tool],
    system_prompt=(
        "You are a careful math assistant. "
        "Use the add_numbers tool whenever a user asks for a sum from free-form text."
    ),
    debug=True,
    name="string_tool_agent",
)

string_tool_result = string_tool_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the sum of 27, 2, and 1?",
            }
        ]
    }
)

print_last_message(string_tool_result)


[values] {'messages': [HumanMessage(content='What is the sum of 27, 2, and 1?', additional_kwargs={}, response_metadata={}, id='5571812f-27ab-47b4-94a7-28e61ed8997d')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 90, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_a65955873c', 'id': 'chatcmpl-DWeuGPBptrQ9s2dhy5WbeFywaXfyg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='string_tool_agent', id='lc_run--019daa26-033b-7712-8a00-c6b634fde5c4-0', tool_calls=[{'name': 'add_numbers', 'args': {'__arg1': '27, 2, and 1'}, 'id': 'call_405JliXtvuJvGxP5X02t8B11', 'type':

In [8]:
structured_agent = create_agent(
    model=llm,
    tools=[add_numbers_with_options],
    system_prompt=(
        "You are a careful math assistant. "
        "When arithmetic is requested, call the provided tool instead of guessing."
    ),
    debug=True,
    name="structured_tool_agent",
)

structured_result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Sum -1.2 and -5.0 using absolute values.",
            }
        ]
    }
)

print_last_message(structured_result)


[values] {'messages': [HumanMessage(content='Sum -1.2 and -5.0 using absolute values.', additional_kwargs={}, response_metadata={}, id='f4e5b45a-5dca-4bb0-a5f4-2053e79c29a2')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 134, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_698dd3b40e', 'id': 'chatcmpl-DWeuKWtxWRyy2xxpFf3OZMcKwKBkV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='structured_tool_agent', id='lc_run--019daa26-15f6-77d2-b891-33f538137a5f-0', tool_calls=[{'name': 'add_numbers_with_options', 'args': {'numbers': [-1.2, -5.0], 'absolute': True}, 'id

In [9]:
class SumAnswer(BaseModel):
    numbers: List[float] = Field(description="The numbers extracted from the request.")
    absolute: bool = Field(description="Whether absolute values were used.")
    result: float = Field(description="The computed sum.")
    explanation: str = Field(description="A short explanation of the calculation.")


openai_structured_agent = create_agent(
    model=llm,
    tools=[add_numbers_with_options],
    system_prompt=(
        "You are a careful math assistant. "
        "Use the tool for calculations and return the final answer in the requested schema."
    ),
    response_format=SumAnswer,
    debug=True,
    name="openai_structured_output_agent",
)

structured_output_result = openai_structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Sum -1.2 and -5.0 using absolute values and explain briefly.",
            }
        ]
    }
)

structured_output_result["structured_response"]


[values] {'messages': [HumanMessage(content='Sum -1.2 and -5.0 using absolute values and explain briefly.', additional_kwargs={}, response_metadata={}, id='be9b1873-1181-46a7-8a8e-30d5718ceb65')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 239, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c6c2d9662', 'id': 'chatcmpl-DWeuVCPMRYdZ3TBsXJ3EAvkv0973q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='openai_structured_output_agent', id='lc_run--019daa26-4216-7a83-8598-684670647fc9-0', tool_calls=[{'name': 'add_numbers_with_options', 'args': {'n

SumAnswer(numbers=[-1.2, -5.0], absolute=True, result=6.2, explanation='The sum was calculated using the absolute values of -1.2 and -5.0, which are 1.2 and 5.0 respectively. Their sum is 6.2.')

## 3. Takeaways

- Simple string tools are easy to write, but brittle.
- Decorated and structured tools expose better schemas to the model.
- Modern LangChain uses a unified agent API, but the same lesson still applies: better tool schemas usually produce more reliable tool calls.
- OpenAI models can handle both tool use and structured final outputs in a single flow.
